# 🇻🇳 ViRecognAgent: Multi-Agent Vietnamese Handwritten Text Recognition

**Paper**: *Beyond Single-Pass OCR: Multi-Agent Diacritic Verification for Vietnamese Handwritten Text*

This notebook runs the complete experimental pipeline:
1. Setup & install dependencies
2. Download Vietnamese handwriting datasets
3. Run baselines (Gemma 4 E4B zero-shot, VietOCR)
4. Run the full confidence-gated multi-agent pipeline
5. Run ablations and threshold sweeps
6. Visualize results with CER, WER, and DER metrics

**Requirements**: Colab GPU runtime (T4 minimum, A100 recommended for Gemma 4)

## 1. Setup — Clone repo and install dependencies

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone the ViRecognAgent repository
GITHUB_REPO = "nmnhut-it/virecognagent"

!rm -rf virecognagent
!git clone https://github.com/{GITHUB_REPO}.git
%cd virecognagent


In [ ]:
# Install dependencies
!pip install -r requirements.txt 2>&1 | tail -40

# Verify key imports
import paddleocr
import vietocr
import transformers
import jiwer
print(f"PaddleOCR: {paddleocr.__version__}")
print(f"Transformers: {transformers.__version__}")
print("All dependencies installed successfully!")


In [ ]:
# Smoke test — verify all modules import and core logic works
!python scripts/test_smoke.py


## 2. Download Datasets

In [ ]:
# Download 5CD-AI/Viet-Handwriting-OCR from HuggingFace
# ~23,403 Vietnamese handwriting images with transcriptions
from datasets import load_dataset

print("Loading 5CD-AI/Viet-Handwriting-OCR...")
dataset = load_dataset("5CD-AI/Viet-Handwriting-OCR")
print(f"\nDataset loaded:")
print(dataset)

In [ ]:
# Explore the dataset
import matplotlib.pyplot as plt
from PIL import Image

# Show sample images
split = list(dataset.keys())[0]
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for idx, ax in enumerate(axes.flat):
    sample = dataset[split][idx]
    img = sample["image"]
    text = sample.get("text", sample.get("label", "N/A"))
    ax.imshow(img)
    ax.set_title(text[:50], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Vietnamese Handwriting Images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Dataset statistics — count diacritics
VIET_DIACRITICS = set(
    "àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợ"
    "ùúủũụưứừửữựỳýỷỹỵđ"
)

split = list(dataset.keys())[0]
texts = [s.get("text", s.get("label", "")) for s in dataset[split]]
total_chars = sum(len(t) for t in texts)
diacritic_chars = sum(1 for t in texts for c in t if c.lower() in VIET_DIACRITICS)

print(f"Split: {split}")
print(f"Total samples: {len(texts):,}")
print(f"Total characters: {total_chars:,}")
print(f"Diacritic characters: {diacritic_chars:,} ({diacritic_chars/total_chars:.1%})")
print(f"Unique characters: {len(set(''.join(texts)))}")
print(f"Avg text length: {total_chars/len(texts):.1f} chars")

In [ ]:
# Create evaluation subset (use first N samples for quick experiments)
# Increase for final paper results
N_EVAL = 100  # ← START SMALL, increase to 500-1000 for paper

import random
random.seed(42)

split = list(dataset.keys())[0]
all_indices = list(range(len(dataset[split])))
random.shuffle(all_indices)
eval_indices = all_indices[:N_EVAL]

eval_samples = [dataset[split][i] for i in eval_indices]
print(f"Evaluation subset: {len(eval_samples)} samples")

## 3. Baseline 1: VietOCR

In [ ]:
from src.recognition import VietOCRRecognizer
from src.metrics import evaluate
from tqdm.notebook import tqdm

# Initialize VietOCR
vietocr = VietOCRRecognizer(device="cuda")

# Run recognition
vietocr_predictions = []
vietocr_references = []

for sample in tqdm(eval_samples, desc="VietOCR baseline"):
    img = sample["image"]
    gt = sample.get("text", sample.get("label", ""))
    
    result = vietocr.recognize(img)
    vietocr_predictions.append(result.text)
    vietocr_references.append(gt)

# Evaluate
vietocr_metrics = evaluate(vietocr_predictions, vietocr_references)
print("\n" + "="*50)
print("BASELINE: VietOCR")
print("="*50)
print(vietocr_metrics)

## 4. Baseline 2: Gemma 4 E4B Zero-Shot

In [ ]:
from src.recognition import Gemma4Recognizer

# Initialize Gemma 4 E4B (4-bit quantized to fit T4)
gemma = Gemma4Recognizer(
    model_name="google/gemma-4-E4B-it",
    quantization="4bit",
    visual_token_budget=1120,  # max detail for handwriting
)

# Run zero-shot recognition (thinking OFF)
gemma_predictions = []
gemma_references = []

for sample in tqdm(eval_samples, desc="Gemma 4 zero-shot"):
    img = sample["image"]
    gt = sample.get("text", sample.get("label", ""))
    
    result = gemma.recognize(img, thinking=False)
    gemma_predictions.append(result.text)
    gemma_references.append(gt)

# Evaluate
gemma_metrics = evaluate(gemma_predictions, gemma_references)
print("\n" + "="*50)
print("BASELINE: Gemma 4 E4B zero-shot (thinking OFF)")
print("="*50)
print(gemma_metrics)

In [ ]:
# Gemma 4 with thinking ON
gemma_think_predictions = []

for sample in tqdm(eval_samples, desc="Gemma 4 thinking ON"):
    img = sample["image"]
    gt = sample.get("text", sample.get("label", ""))
    
    result = gemma.recognize(img, thinking=True)
    gemma_think_predictions.append(result.text)

gemma_think_metrics = evaluate(gemma_think_predictions, gemma_references)
print("\n" + "="*50)
print("BASELINE: Gemma 4 E4B zero-shot (thinking ON)")
print("="*50)
print(gemma_think_metrics)

## 5. Full Pipeline: Confidence-Gated Multi-Agent

In [ ]:
from src.pipeline import ViRecognAgentPipeline

# Initialize full pipeline
pipeline = ViRecognAgentPipeline(
    mode="full",
    confidence_threshold=0.85,
    max_rounds=2,
    gemma_quantization="4bit",
)

# Run pipeline
pipeline_predictions = []
pipeline_references = []
pipeline_details = []

for sample in tqdm(eval_samples, desc="Full pipeline"):
    img = sample["image"]
    gt = sample.get("text", sample.get("label", ""))
    
    line_result = pipeline.process_single_line(img)
    pipeline_predictions.append(line_result.text)
    pipeline_references.append(gt)
    pipeline_details.append(line_result)

# Evaluate
pipeline_metrics = evaluate(pipeline_predictions, pipeline_references)

# Agent activation stats
n_activated = sum(1 for d in pipeline_details if d.agent_activated)
activation_rate = n_activated / len(pipeline_details)

print("\n" + "="*50)
print("FULL PIPELINE: VietOCR + Confidence Gate + Agent Pod")
print("="*50)
print(pipeline_metrics)
print(f"\nAgent activation rate: {activation_rate:.1%} ({n_activated}/{len(pipeline_details)} lines)")

## 6. Compare All Results

In [ ]:
import pandas as pd

results = {
    "VietOCR": vietocr_metrics,
    "Gemma4 (no think)": gemma_metrics,
    "Gemma4 (think)": gemma_think_metrics,
    "Full Pipeline": pipeline_metrics,
}

df = pd.DataFrame({
    name: {
        "CER ↓": f"{m.cer:.4f}",
        "WER ↓": f"{m.wer:.4f}",
        "DER ↓": f"{m.der:.4f}",
        "Tone ER ↓": f"{m.tone_er:.4f}",
    }
    for name, m in results.items()
}).T

print("\n" + "="*70)
print("COMPARISON TABLE (for paper)")
print("="*70)
print(df.to_string())
print()

# Save as CSV for paper tables
df.to_csv("outputs/comparison_table.csv")
print("Saved to outputs/comparison_table.csv")

In [ ]:
# Visualization: bar chart comparing metrics
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

methods = list(results.keys())
metrics_to_plot = [
    ("CER", [r.cer for r in results.values()]),
    ("WER", [r.wer for r in results.values()]),
    ("DER", [r.der for r in results.values()]),
]

colors = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63"]

for ax, (metric_name, values) in zip(axes, metrics_to_plot):
    bars = ax.bar(methods, values, color=colors[:len(methods)])
    ax.set_title(metric_name, fontsize=14, fontweight="bold")
    ax.set_ylabel("Error Rate")
    ax.tick_params(axis="x", rotation=30)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", fontsize=9)

plt.suptitle("ViRecognAgent: Vietnamese HTR Comparison", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved chart to outputs/comparison_chart.png")

## 7. Threshold Sweep (for paper Figure)

In [ ]:
# Sweep confidence threshold from 0.5 to 0.95
thresholds = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]
sweep_results = []

for threshold in thresholds:
    print(f"\nRunning threshold={threshold}...")
    pipe = ViRecognAgentPipeline(
        mode="full",
        confidence_threshold=threshold,
        max_rounds=2,
        gemma_quantization="4bit",
    )
    
    preds, refs, details = [], [], []
    for sample in tqdm(eval_samples, desc=f"t={threshold}", leave=False):
        img = sample["image"]
        gt = sample.get("text", sample.get("label", ""))
        line_result = pipe.process_single_line(img)
        preds.append(line_result.text)
        refs.append(gt)
        details.append(line_result)
    
    m = evaluate(preds, refs)
    act_rate = sum(1 for d in details if d.agent_activated) / len(details)
    sweep_results.append({
        "threshold": threshold,
        "cer": m.cer, "wer": m.wer, "der": m.der,
        "activation_rate": act_rate,
    })
    print(f"  CER={m.cer:.4f} DER={m.der:.4f} Agent%={act_rate:.1%}")

print("\nThreshold sweep complete!")

In [ ]:
# Plot threshold sweep
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ts = [r["threshold"] for r in sweep_results]

ax1.plot(ts, [r["cer"] for r in sweep_results], "o-", label="CER", color="#2196F3")
ax1.plot(ts, [r["der"] for r in sweep_results], "s-", label="DER", color="#E91E63")
ax1.set_xlabel("Confidence Threshold")
ax1.set_ylabel("Error Rate")
ax1.set_title("Accuracy vs. Confidence Threshold")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ts, [r["activation_rate"] for r in sweep_results], "D-", color="#4CAF50")
ax2.set_xlabel("Confidence Threshold")
ax2.set_ylabel("Agent Activation Rate")
ax2.set_title("Agent Pod Activation vs. Threshold")
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.suptitle("Confidence Threshold Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Error Analysis — Where does the Agent Pod help?

In [ ]:
# Show examples where agent correction fixed diacritic errors
print("Examples where Agent Pod corrected diacritics:")
print("=" * 70)

n_shown = 0
for i, detail in enumerate(pipeline_details):
    if detail.agent_activated and n_shown < 10:
        gt = pipeline_references[i]
        pred = pipeline_predictions[i]
        print(f"\nSample {i}:")
        print(f"  Ground truth: {gt}")
        print(f"  Final output: {pred}")
        print(f"  Method:       {detail.method}")
        print(f"  Confidence:   {detail.confidence:.3f}")
        if detail.diacritics:
            print(f"  Diacritics:   {detail.diacritics.count} chars, risk={detail.diacritics.der_risk}")
        if detail.agent_trace:
            print(f"  Agent rounds: {detail.agent_trace.rounds}")
            print(f"    Initial:    {detail.agent_trace.initial_text}")
            print(f"    Proposer:   {detail.agent_trace.proposer_text}")
        print(f"  Match: {'✓' if gt == pred else '✗'}")
        n_shown += 1

# Show structured JSON output for one example
if pipeline_details:
    print("\n" + "=" * 70)
    print("STRUCTURED OUTPUT EXAMPLE (JSON):")
    print("=" * 70)
    import json
    example = pipeline_details[0].to_dict()
    print(json.dumps(example, ensure_ascii=False, indent=2))

## 9. Save All Results

In [ ]:
import json
import os

os.makedirs("outputs", exist_ok=True)

# Save complete results
all_results = {
    "baselines": {
        "vietocr": vietocr_metrics.to_dict(),
        "gemma4_zeroshot": gemma_metrics.to_dict(),
        "gemma4_thinking": gemma_think_metrics.to_dict(),
    },
    "full_pipeline": pipeline_metrics.to_dict(),
    "agent_activation_rate": activation_rate,
    "threshold_sweep": sweep_results,
    "config": {
        "n_eval_samples": N_EVAL,
        "confidence_threshold": 0.85,
        "max_rounds": 2,
        "model": "google/gemma-4-E4B-it",
        "quantization": "4bit",
    }
}

with open("outputs/full_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("All results saved to outputs/full_results.json")
print("\nFiles in outputs/:")
for f in sorted(os.listdir("outputs")):
    print(f"  {f}")

---

## Next Steps

1. **Increase N_EVAL** to 500-1000 for paper-quality results
2. **Run ablations**: change `mode` to `ablation_no_linguist` or `ablation_no_judge`
3. **Try different datasets**: download Cinnamon AI or request VNOnDB access
4. **Fine-tune with LoRA** (future work — see `scripts/finetune_lora.py`)
5. **Generate paper figures** from `outputs/` directory